In [1]:
!git clone https://github.com/ycwsorry/maize-disease-detection-dataset.git /kaggle/working/maize-disease-detection


Cloning into '/kaggle/working/maize-disease-detection'...
remote: Enumerating objects: 18853, done.
remote: Total 18853 (delta 0), reused 0 (delta 0), pack-reused 18853 (from 3)
Receiving objects: 100% (18853/18853), 879.23 MiB | 35.80 MiB/s, done.
Resolving deltas: 100% (305/305), done.
Updating files: 100% (30252/30252), done.


In [2]:
from pathlib import Path
import zipfile

# Uploaded extracted weight
WEIGHT_DIR = Path(
    "/kaggle/input/datasets/siam2461/weight/best"
)

# Output .pt
RECONSTRUCTED_PT = Path(
    "/kaggle/working/reconstructed_best.pt"
)

print("Source exists:", WEIGHT_DIR.exists())
if RECONSTRUCTED_PT.exists():
    RECONSTRUCTED_PT.unlink()
    
with zipfile.ZipFile(
    RECONSTRUCTED_PT,
    mode="w",
    compression=zipfile.ZIP_STORED
) as zf:

    for file_path in WEIGHT_DIR.rglob("*"):

        if file_path.is_file():

            relative_path = file_path.relative_to(
                WEIGHT_DIR
            )

            # Put everything inside "archive/"
            arcname = Path("archive") / relative_path
            zf.write(
                file_path,
                arcname=str(arcname)
            )

print("=" * 70)
print("CORRECTED RECONSTRUCTION COMPLETE")
print("=" * 70)

print("Path:")
print(RECONSTRUCTED_PT)
print(
    "Size:",
    f"{RECONSTRUCTED_PT.stat().st_size / (1024**2):.2f} MB"
)
print("\nArchive structure:")

with zipfile.ZipFile(RECONSTRUCTED_PT, "r") as zf:

    for name in zf.namelist()[:20]:
        print(name)

print("=" * 70)

Source exists: True
CORRECTED RECONSTRUCTION COMPLETE
Path:
/kaggle/working/reconstructed_best.pt
Size: 5.22 MB

Archive structure:
archive/.format_version
archive/.storage_alignment
archive/byteorder
archive/data.pkl
archive/version
archive/.data/serialization_id
archive/data/0
archive/data/1
archive/data/10
archive/data/100
archive/data/101
archive/data/102
archive/data/103
archive/data/104
archive/data/105
archive/data/106
archive/data/107
archive/data/108
archive/data/109
archive/data/11


In [3]:
!pip install -q ultralytics==8.4.136

In [4]:
from pathlib import Path

ROOT = "/kaggle/working/maize-disease-detection"

Path("/kaggle/working/val_test.yaml").write_text(f"""path: {ROOT}
train: images/train
val: images/val
test: images/test
names:
  0: Blight
  1: brown_spot
  2: corn_rust
  3: corn_smut
  4: downy_mildew
  5: grey_leaf_spot
  6: healthy
  7: maize-streak-disease
""")

print(open("/kaggle/working/val_test.yaml").read())

path: /kaggle/working/maize-disease-detection
train: images/train
val: images/val
test: images/test
names:
  0: Blight
  1: brown_spot
  2: corn_rust
  3: corn_smut
  4: downy_mildew
  5: grey_leaf_spot
  6: healthy
  7: maize-streak-disease



In [5]:
from ultralytics import YOLO

model = YOLO("/kaggle/working/reconstructed_best.pt")

metrics = model.val(
    data="/kaggle/working/val_test.yaml",
    split="test",
    imgsz=640,
    batch=1,
    device=0,
)

print(metrics.speed)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Ultralytics 8.4.136 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11n summary (fused): 101 layers, 2,583,712 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1203.2±352.1 MB/s, size: 48.3 KB)
val: Scanning /kaggle/working/maize-disease-detection/labels/test... 1506 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1506/1506 1.3Kit/s 1.2s
val: New cache created: /kaggle/working/maize-disease-detection/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1506/1506 58.4it/s 25.8s
                   all       1506       2947      0.852      0.819      0.

In [6]:
import time, statistics, torch
from pathlib import Path
from PIL import Image
from transformers import AutoImageProcessor, AutoModelForDepthEstimation

MODEL = "depth-anything/Depth-Anything-V2-Small-hf"
TEST  = Path("/kaggle/working/maize-disease-detection/images/test")

proc = AutoImageProcessor.from_pretrained(MODEL)
dm   = AutoModelForDepthEstimation.from_pretrained(MODEL).to("cuda").eval()

imgs = sorted(p for p in TEST.iterdir() if p.suffix.lower() in {".jpg",".jpeg",".png"})[:110]

times = []
for i, p in enumerate(imgs):
    im = Image.open(p).convert("RGB")
    torch.cuda.synchronize(); t0 = time.perf_counter()
    batch = proc(images=im, return_tensors="pt").to("cuda")
    with torch.no_grad():
        _ = dm(**batch).predicted_depth
    torch.cuda.synchronize(); t1 = time.perf_counter()
    if i >= 10:                      # first 10 discarded as warm-up
        times.append((t1 - t0) * 1000)

print(f"n = {len(times)}")
print(f"mean   = {statistics.mean(times):.1f} ms")
print(f"SD     = {statistics.stdev(times):.1f} ms")
print(f"median = {statistics.median(times):.1f} ms")

preprocessor_config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/950 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 99.2MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/287 [00:00<?, ?it/s]

n = 100
mean   = 43.6 ms
SD     = 0.9 ms
median = 43.5 ms


In [7]:
from pathlib import Path
import zipfile

# Uploaded extracted weight
WEIGHT_DIR = Path(
    "/kaggle/input/datasets/siam2461/new-weight/best/best"
)

# Output .pt
RECONSTRUCTED_PT = Path(
    "/kaggle/working/reconstructed_best.pt"
)

print("Source exists:", WEIGHT_DIR.exists())
if RECONSTRUCTED_PT.exists():
    RECONSTRUCTED_PT.unlink()
    
with zipfile.ZipFile(
    RECONSTRUCTED_PT,
    mode="w",
    compression=zipfile.ZIP_STORED
) as zf:

    for file_path in WEIGHT_DIR.rglob("*"):

        if file_path.is_file():

            relative_path = file_path.relative_to(
                WEIGHT_DIR
            )

            # Put everything inside "archive/"
            arcname = Path("archive") / relative_path
            zf.write(
                file_path,
                arcname=str(arcname)
            )

print("=" * 70)
print("CORRECTED RECONSTRUCTION COMPLETE")
print("=" * 70)

print("Path:")
print(RECONSTRUCTED_PT)
print(
    "Size:",
    f"{RECONSTRUCTED_PT.stat().st_size / (1024**2):.2f} MB"
)
print("\nArchive structure:")

with zipfile.ZipFile(RECONSTRUCTED_PT, "r") as zf:

    for name in zf.namelist()[:20]:
        print(name)

print("=" * 70)

Source exists: True
CORRECTED RECONSTRUCTION COMPLETE
Path:
/kaggle/working/reconstructed_best.pt
Size: 5.22 MB

Archive structure:
archive/.format_version
archive/.storage_alignment
archive/byteorder
archive/data.pkl
archive/version
archive/.data/serialization_id
archive/data/0
archive/data/1
archive/data/10
archive/data/100
archive/data/101
archive/data/102
archive/data/103
archive/data/104
archive/data/105
archive/data/106
archive/data/107
archive/data/108
archive/data/109
archive/data/11


In [8]:
from pathlib import Path

ROOT = "/kaggle/working/maize-disease-detection"

Path("/kaggle/working/val_test.yaml").write_text(f"""path: {ROOT}
train: images/train
val: images/val
test: images/test
names:
  0: Blight
  1: brown_spot
  2: corn_rust
  3: corn_smut
  4: downy_mildew
  5: grey_leaf_spot
  6: healthy
  7: maize-streak-disease
""")

print(open("/kaggle/working/val_test.yaml").read())

path: /kaggle/working/maize-disease-detection
train: images/train
val: images/val
test: images/test
names:
  0: Blight
  1: brown_spot
  2: corn_rust
  3: corn_smut
  4: downy_mildew
  5: grey_leaf_spot
  6: healthy
  7: maize-streak-disease



In [9]:
from ultralytics import YOLO

model = YOLO("/kaggle/working/reconstructed_best.pt")

metrics = model.val(
    data="/kaggle/working/val_test.yaml",
    split="test",
    imgsz=640,
    batch=1,
    device=0,
)

print(metrics.speed)

Ultralytics 8.4.136 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11n summary (fused): 101 layers, 2,583,712 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1943.6±877.5 MB/s, size: 76.6 KB)
val: Scanning /kaggle/working/maize-disease-detection/labels/test.cache... 1506 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1506/1506 451.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1506/1506 62.3it/s 24.2s
                   all       1506       2947      0.852      0.819      0.885      0.691
                Blight        205        428      0.902       0.89      0.927      0.685
            brown_spot        146        347      0.805      0.847      0.902      0.596
             corn_rust        238        503      0.887      0.624      0.782      0.569
             corn_smut        239        395      0.773      0.818      0.868      0.692
          dow

In [10]:
from pathlib import Path
import zipfile

# Uploaded extracted weight
WEIGHT_DIR = Path(
    "/kaggle/input/datasets/siam2461/new-weight/last/last"
)

# Output .pt
RECONSTRUCTED_PT = Path(
    "/kaggle/working/reconstructed_best.pt"
)

print("Source exists:", WEIGHT_DIR.exists())
if RECONSTRUCTED_PT.exists():
    RECONSTRUCTED_PT.unlink()
    
with zipfile.ZipFile(
    RECONSTRUCTED_PT,
    mode="w",
    compression=zipfile.ZIP_STORED
) as zf:

    for file_path in WEIGHT_DIR.rglob("*"):

        if file_path.is_file():

            relative_path = file_path.relative_to(
                WEIGHT_DIR
            )

            # Put everything inside "archive/"
            arcname = Path("archive") / relative_path
            zf.write(
                file_path,
                arcname=str(arcname)
            )

print("=" * 70)
print("CORRECTED RECONSTRUCTION COMPLETE")
print("=" * 70)

print("Path:")
print(RECONSTRUCTED_PT)
print(
    "Size:",
    f"{RECONSTRUCTED_PT.stat().st_size / (1024**2):.2f} MB"
)
print("\nArchive structure:")

with zipfile.ZipFile(RECONSTRUCTED_PT, "r") as zf:

    for name in zf.namelist()[:20]:
        print(name)

print("=" * 70)

Source exists: True
CORRECTED RECONSTRUCTION COMPLETE
Path:
/kaggle/working/reconstructed_best.pt
Size: 5.22 MB

Archive structure:
archive/.format_version
archive/.storage_alignment
archive/byteorder
archive/data.pkl
archive/version
archive/.data/serialization_id
archive/data/0
archive/data/1
archive/data/10
archive/data/100
archive/data/101
archive/data/102
archive/data/103
archive/data/104
archive/data/105
archive/data/106
archive/data/107
archive/data/108
archive/data/109
archive/data/11


In [11]:
from pathlib import Path

ROOT = "/kaggle/working/maize-disease-detection"

Path("/kaggle/working/val_test.yaml").write_text(f"""path: {ROOT}
train: images/train
val: images/val
test: images/test
names:
  0: Blight
  1: brown_spot
  2: corn_rust
  3: corn_smut
  4: downy_mildew
  5: grey_leaf_spot
  6: healthy
  7: maize-streak-disease
""")

print(open("/kaggle/working/val_test.yaml").read())

path: /kaggle/working/maize-disease-detection
train: images/train
val: images/val
test: images/test
names:
  0: Blight
  1: brown_spot
  2: corn_rust
  3: corn_smut
  4: downy_mildew
  5: grey_leaf_spot
  6: healthy
  7: maize-streak-disease



In [12]:
from ultralytics import YOLO

model = YOLO("/kaggle/working/reconstructed_best.pt")

metrics = model.val(
    data="/kaggle/working/val_test.yaml",
    split="test",
    imgsz=640,
    batch=1,
    device=0,
)

print(metrics.speed)

Ultralytics 8.4.136 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11n summary (fused): 101 layers, 2,583,712 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1557.4±608.9 MB/s, size: 57.4 KB)
val: Scanning /kaggle/working/maize-disease-detection/labels/test.cache... 1506 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1506/1506 451.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1506/1506 61.5it/s 24.5s
                   all       1506       2947      0.852      0.819      0.885      0.691
                Blight        205        428      0.902       0.89      0.927      0.685
            brown_spot        146        347      0.805      0.847      0.902      0.596
             corn_rust        238        503      0.887      0.624      0.782      0.569
             corn_smut        239        395      0.773      0.818      0.868      0.692
          dow